# 🐍 Python Deep Theory & Tricky Gotchas
### *Everything that can surprise you on a test — built from your PPY lecture notes + examples*

---

Each section has:
- 📖 **Bold theory** explaining *why* it happens internally
- 💻 **Runnable code cells** you can play with and modify
- ⚠️ **Exam traps** highlighted clearly

**Topics:**
1. Objects & References — `is` vs `==`, shallow vs deep copy
2. Integer & String Caching / Interning
3. Mutable Default Arguments
4. Default Argument Evaluation Time
5. Scopes: LEGB, `global`, `nonlocal`, UnboundLocalError
6. Closures & Late Binding
7. Generators — `yield`, send(), priming, two-way communication
8. Decorator Mechanics & Stacking Order
9. Parametrized Decorators
10. `__new__` vs `__init__`
11. Name Mangling (`__private`)
12. Magic Methods: `__eq__`, `__hash__`, `__bool__`, `__call__`, `__radd__`
13. `@functools.total_ordering`
14. `@singledispatch` dispatch rules (bool→int, subclass→parent)
15. Exception `else` clause and using exceptions as control flow
16. Boolean `and`/`or` — they return *values*, not booleans
17. `for`/`while` `else` clause
18. Chained comparisons
19. List multiplication — references, not copies
20. Extended unpacking — starred assignment
21. `*args` / `**kwargs` packing and unpacking
22. Boolean arithmetic — `True=1`, `False=0`
23. Pull-pipe vs Push-pipe generator patterns
24. Properties — getters, setters, validation
25. MRO — Multiple Inheritance Resolution
26. 🎯 Quick-fire exam traps

---
## 1. 🔗 Objects & References — `is` vs `==`, Shallow vs Deep Copy

### Theory

**In Python, variables are NOT boxes that hold values. They are *labels* (references) pointing to objects in memory.**

- **`==`** calls `__eq__` — asks: *do these objects have the same value?*
- **`is`** checks object identity — asks: *are these literally the same object in memory?* (same `id()`)

**Assignment `b = a` does NOT copy the object — it creates a second label pointing to the same object.**

**Shallow copy** (`list(a)`, `a[:]`, `a.copy()`):
- Creates a **new outer container**
- Elements inside are **still the same objects** — copies of references, not copies of objects
- ⚠️ **Mutating a nested mutable (like an inner list) through either variable affects BOTH!**

**Deep copy** (`copy.deepcopy(a)`):
- Creates a **completely independent copy**, recursively duplicating every nested object
- Mutations to one have **zero effect** on the other

In [ ]:
from copy import deepcopy

# --- Assignment = same object ---
a = [1, 2, [3, 5]]
b = a                     # b is just another name for a!
print('b is a:', b is a)           # True  — same object
a[2][1] = 4
print('b after a[2][1]=4:', b)     # [1, 2, [3, 4]]  — b sees the change!
print()

# --- Shallow copy ---
c = list(a)               # shallow copy — new outer list
c.append(5)
print('c is a:', c is a)           # False — different outer list
print('c[2] is a[2]:', c[2] is a[2])   # True — SAME inner list!
a[2][1] = 6               # mutate inner list through 'a'
print('a:', a)             # [1, 2, [3, 6]]
print('c:', c)             # [1, 2, [3, 6], 5]  — c sees the change in inner list!
print()

# --- Deep copy ---
d = deepcopy(a)
print('d[2] is a[2]:', d[2] is a[2])   # False — different inner list
a[2][1] = 7
print('a:', a)             # [1, 2, [3, 7]]
print('d:', d)             # [1, 2, [3, 6], 5]  — d is unaffected!

---
## 2. 🔢 Integer & String Caching (Interning)

### Theory

**CPython pre-creates integer objects for the range `-5` to `256` at startup and reuses them.** Writing `a = 100` returns a reference to the already-existing cached object.

**For integers outside this range** (e.g. `257`), a new object is typically created — so two variables with the same value may or may not be the same object.

⚠️ **This is context-dependent!** Inside a single compiled block (e.g., a script file), Python may intern larger integers as an optimisation. In the REPL, each line compiles separately — so `257 is 257` on two separate lines returns `False`.

**String interning**: Python automatically interns strings that look like identifiers (no spaces, no special chars). Strings with spaces or dynamically created strings may NOT be interned.

**Golden rule: always use `==` for value comparison. Use `is` ONLY for `None`, `True`, `False`, and explicit singletons.**

In [ ]:
# Small integers — CACHED (-5 to 256)
a, b = 256, 256
print(f'256 is 256 → {a is b}')    # True  (cached)
print(f'256 == 256 → {a == b}')    # True

# Large integers — NOT guaranteed cached
x, y = 257, 257
print(f'257 is 257 → {x is y}')    # May be True in script, False in REPL
print(f'257 == 257 → {x == y}')    # Always True

print()

# None is ALWAYS a singleton
print(f'None is None → {None is None}')  # Always True

# True and False are singletons
print(f'True is True → {True is True}')   # True
print(f'True == 1 → {True == 1}')         # True  (bool subclass of int)
print(f'True is 1 → {True is 1}')         # False! Different objects

print()

# String interning
s1, s2 = 'hello', 'hello'
print(f'"hello" is "hello" → {s1 is s2}')   # True (interned)

s3 = 'hello world'
s4 = 'hello world'
print(f'"hello world" is "hello world" → {s3 is s4}')  # May vary!

# id() shows memory address
print(f'id(100) == id(100): {id(100) == id(100)}')  # True — same cached object

---
## 3. ⚡ Mutable Default Arguments — The Classic Trap

### Theory

**Default argument values are evaluated ONCE — at function DEFINITION time — not each time the function is called.**

If the default is a **mutable object** (list, dict, set), it is **shared across ALL calls** to that function.

- First call: appends to `[]` → list becomes `[1]`
- Second call (no argument): the default IS that same `[1]` list → becomes `[1, 2]`
- **The list grows across calls and is never reset!**

**Fix**: use `None` as the default and create a new mutable object inside the function body.

⚠️ **Related trap**: `def f(b=i)` — the default value of `b` is captured at *definition* time. Even if `i` changes later, `b`'s default does NOT change.

In [ ]:
# THE TRAP — mutable default list
def funlist(val, lst=[]):   # lst=[] created ONCE at definition time!
    lst.append(val)
    return lst

print(funlist(1))               # [1]
print(funlist(2))               # [1, 2]  — NOT [2]!
print(funlist(3))               # [1, 2, 3]
print(funlist(4))               # [1, 2, 3, 4]
print(funlist('c', ['a','b']))  # ['a', 'b', 'c']  — explicit arg is fine
print(funlist('d'))             # [1, 2, 3, 4, 'd']  — still the growing list!
print('\nActual default object:', funlist.__defaults__)

In [ ]:
# THE FIX — use None as sentinel
def funlist_fixed(val, lst=None):
    if lst is None:
        lst = []       # New list created EACH call
    lst.append(val)
    return lst

print(funlist_fixed(1))   # [1]
print(funlist_fixed(2))   # [2]  ✓ correct!
print(funlist_fixed(3))   # [3]  ✓
print()

# RELATED: default value captured at definition time
i = 7
def fundef(a, b=i):   # b's default = 7, captured NOW
    return a + b

i += 10   # i is now 17, but b's default is STILL 7!
print('i =', i, '  fundef(1) =', fundef(1))  # i=17, fundef(1)=8  (1+7 not 1+17)

---
## 4. 🌍 Scopes: LEGB, `global`, `nonlocal`, UnboundLocalError

### Theory

**Python looks up variable names using the LEGB rule:**
1. **L**ocal — inside the current function
2. **E**nclosing — in any surrounding (nested) function scopes, inside-out
3. **G**lobal — at module level
4. **B**uilt-in — Python's built-in names (`len`, `print`, `range`, etc.)

**The UnboundLocalError trap:**
**If Python sees ANY assignment to `x` anywhere inside a function, it treats `x` as LOCAL for the ENTIRE function** — even lines BEFORE the assignment. Trying to read it before the assignment causes `UnboundLocalError`.

**`global x`** — declares `x` refers to the module-level variable. Reading AND assigning will affect the global.

**`nonlocal x`** — in a nested function, declares `x` refers to the variable in the nearest *enclosing non-global* scope. Allows the inner function to reassign it.

⚠️ **Without `nonlocal`, inner functions can *read* enclosing variables but CANNOT *rebind* them (augmented assignment like `x += 1` counts as rebinding and will UnboundLocalError).**

In [ ]:
# UnboundLocalError trap
x = 10

def bad_func():
    # Python sees 'x += 1' below → treats x as LOCAL for whole function
    # But x is read before the assignment → UnboundLocalError!
    try:
        print(x)   # ← UnboundLocalError here
        x += 1
    except UnboundLocalError as e:
        print('UnboundLocalError:', e)

bad_func()
print('Global x unchanged:', x)
print()

# Fix with 'global'
counter = 0
def increment():
    global counter
    counter += 1

increment(); increment(); increment()
print('counter =', counter)  # 3
print()

# 'nonlocal' — modifying enclosing scope variable
n_global = 3.0

def halves():
    n = -0.5           # local n in halves()
    def addhalf():
        nonlocal n     # refers to halves()'s n, NOT n_global
        n += 0.5
        return n
    return addhalf

f = halves()
print(f())   # 0.0   (-0.5 + 0.5)
print(f())   # 0.5   (0.0 + 0.5)
print('Global n_global still:', n_global)  # 3.0 — untouched!
print('Current closure value:', f.__closure__[0].cell_contents)  # 0.5

---
## 5. 🔒 Closures & Late Binding

### Theory

**A closure is a function that remembers variables from its enclosing scope, even after that scope has finished executing.**

**`func.__closure__`** — a tuple of *cell objects*, one per closed-over variable. Multiple inner functions sharing the same variable share the **same cell** — they all see the current value of that variable.

**Late binding** — the most famous Python gotcha:

**Closures look up variable values at CALL TIME, not at function DEFINITION TIME.**

⚠️ **Classic trap:**
```python
funcs = [lambda x: i * x for i in range(5)]
# All 5 lambdas close over the SAME variable i
# By call time the loop finished → i = 4
# ALL produce x*4, not x*0, x*1, x*2, x*3, x*4!
```

**Fix:** Use a default argument to capture the value immediately:
```python
funcs = [lambda x, i=i: i * x for i in range(5)]  # i=i captures current value of i
```

In [ ]:
# LATE BINDING TRAP
bad_funcs = [lambda x: i * x for i in range(5)]
print('Late binding (all use final i=4):')
print([f(2) for f in bad_funcs])  # [8, 8, 8, 8, 8]  NOT [0, 2, 4, 6, 8]!
print()

# FIX: default argument captures current value
good_funcs = [lambda x, i=i: i * x for i in range(5)]
print('Early binding via default arg:')
print([f(2) for f in good_funcs])  # [0, 2, 4, 6, 8]  ✓
print()

# CLOSURE INTERNALS — __closure__ is a tuple of 'cell' objects
def outer():
    a = 10
    b = 20
    def innA(arg): return arg + a         # closes over only a
    def innB(arg): return arg + b         # closes over only b
    def innC(arg): return arg + a + b     # closes over BOTH a and b
    return innA, innB, innC

fA, fB, fC = outer()
print(f'fA closure length: {len(fA.__closure__)}')  # 1
print(f'fB closure length: {len(fB.__closure__)}')  # 1
print(f'fC closure length: {len(fC.__closure__)}')  # 2
print(f'fA closes over a = {fA.__closure__[0].cell_contents}')   # 10
print(f'fB closes over b = {fB.__closure__[0].cell_contents}')   # 20
print(f'fC cells: {fC.__closure__[0].cell_contents}, {fC.__closure__[1].cell_contents}')  # 10, 20

---
## 6. ⚙️ Generators — `yield`, Execution Model, Two-Way Communication

### Theory

**A generator function contains a `yield` statement. Calling it does NOT execute the body — it returns a generator object.**

**Execution model:**
- `next(gen)` — resumes execution from where it was suspended, runs until the next `yield`, yields the value, then **suspends again**
- Local state (variables, instruction pointer) is **preserved between calls**
- When the function returns or falls off the end → `StopIteration` is raised

**Two-way communication with `send()`:**
- `x = yield value` — yields `value` outward AND waits to receive the next `send()` value into `x`
- `gen.send(val)` — resumes the generator and makes the `yield` expression evaluate to `val`
- **Priming**: before `send(non-None)`, you MUST advance to the first `yield` via `next(gen)` or `gen.send(None)`

⚠️ **`send(val)` before priming raises `TypeError: can't send non-None value to a just-started generator`**

⚠️ **Generators are exhaustible:** once `StopIteration` fires, iterating again yields nothing.

In [ ]:
# BASIC GENERATOR
def fibonacci(n):
    curr, prev, count = 0, 1, 0
    while count <= n:
        yield curr
        prev, curr = curr, prev + curr
        count += 1

gen = fibonacci(7)
print('Type of function:', type(fibonacci))    # <class 'function'>
print('Type of generator:', type(gen))         # <class 'generator'>
print('First 8 Fibonacci:', list(gen))

# Generator is exhausted!
print('After exhaustion:', list(gen))   # []  — empty!

In [ ]:
# TWO-WAY GENERATOR — send() and priming
def twoWayGener():
    print('Gene: started')
    value = None
    while True:
        x = yield value            # YIELDS value OUT, RECEIVES x IN
        print(f'  Gene: received x={x}, computing value=2*{x}')
        value = 2 * x

gen = twoWayGener()

# PRIMING — advance to first yield before sending
ret = next(gen)          # or gen.send(None)
print(f'After priming: ret={ret}\n')   # None (value was None)

for n in [1, 3, 5]:
    ret = gen.send(n)
    print(f'  Main: sent {n}, got back {ret}\n')

In [ ]:
import functools

# PRIMING DECORATOR — automatically advances to first yield
def coroutine(func):
    @functools.wraps(func)
    def primed(*args, **kwargs):
        gen = func(*args, **kwargs)  # create generator
        next(gen)                    # prime it automatically
        return gen
    return primed

@coroutine
def running_sum():
    total = 0
    while True:
        n = yield total
        total += n

rs = running_sum()   # Already primed — no next() needed!
print(rs.send(10))   # 10
print(rs.send(5))    # 15
print(rs.send(3))    # 18

print()

# CLOSING a generator — sends GeneratorExit inside
@coroutine
def echo():
    while True:
        val = yield
        print('Echo:', val)

e = echo()
e.send('hello')
e.close()

try:
    e.send('world')
except StopIteration:
    print('Generator is closed — StopIteration raised')

---
## 7. 🎨 Decorator Mechanics, Stacking Order & Execution Time

### Theory

**A decorator is a callable that takes a function and returns a replacement function.** `@decorator` is syntactic sugar:
```python
@decorator
def func(): ...
# exactly equivalent to:
func = decorator(func)
```

**CRITICAL timing rule — decoration happens at MODULE LOAD TIME:**
When Python encounters `@decorator` during import/execution, it **immediately calls `decorator(func)`**. Any `print` or code in the *outer* decorator body runs then. The *wrapper* runs only when you actually call `func()`.

**Stacking:**
```python
@A
@B
def func(): ...
# equivalent to: func = A(B(func))
# B applied first (bottom-up), A is outer wrapper (executes first at call time)
```

⚠️ **Order matters: `@italic @strong` produces `<i><strong>...</strong></i>` but `@strong @italic` produces `<strong><i>...</i></strong>`.**

**`@functools.wraps(func)`** — preserves `__name__`, `__doc__` of the original. Without it, the wrapper hides the function's identity.

In [ ]:
import functools

def decorator(func):
    print(f'*** Decorating {func.__name__}')   # Runs at DECORATION TIME (module load)
    @functools.wraps(func)
    def wrap(arg):
        val = func(arg)
        print(f'Decorated {func.__name__}: {arg} -> {val}')  # Runs at CALL TIME
        return val
    return wrap

print('--- Defining decorated functions ---')

@decorator
def funA(x): return x

@decorator
def funB(x): return x * x

print('\n--- Now calling them ---')
funA(4)   # 4 -> 4
funB(5)   # 5 -> 25

In [ ]:
# DECORATOR STACKING ORDER
def italic(func):
    @functools.wraps(func)
    def _f(s): return '<i>' + func(s) + '</i>'
    return _f

def strong(func):
    @functools.wraps(func)
    def _f(s): return '<strong>' + func(s) + '</strong>'
    return _f

@italic
@strong
def itst(s): return s     # itst = italic(strong(s))

@strong
@italic
def stit(s): return s     # stit = strong(italic(s))

print(itst(" hello "))    # <i><strong> hello </strong></i>  — italic is OUTER
print(stit(" hello "))    # <strong><i> hello </i></strong>  — strong is OUTER

In [ ]:
# MEMOIZATION — decorator with state stored as function attribute
def memo(func):
    @functools.wraps(func)
    def memoized(*args):
        if args not in memoized.cache:
            memoized.cache[args] = func(*args)
        return memoized.cache[args]
    memoized.cache = {}   # Attribute on the wrapper itself!
    return memoized

@memo
def fibo(n):
    if n < 2: return n
    return fibo(n-1) + fibo(n-2)

print(fibo(100))   # Fast!
print('Cache size:', len(fibo.cache))   # Access cache as attribute

---
## 8. 🏭 Parametrized Decorators — 3-Level Nesting

### Theory

**A parametrized decorator is a function that RETURNS a decorator.** Three levels of nesting:

```
rangecheck(mn, mx)     ← called with parameters → returns a decorator
  checker(func)        ← actual decorator → receives the function
    wrap(*args)        ← wrapper → runs on each call
```

`@rangecheck(0, 255)` evaluates `rangecheck(0, 255)` first (getting the decorator), then applies that to the function.

Equivalent to: `color = rangecheck(0, 255)(color)`

In [ ]:
def rangecheck(mn, mx):
    def checker(func):               # actual decorator
        @functools.wraps(func)
        def wrap(*args):             # wrapper — runs each call
            for i in args:
                if not (mn <= i <= mx):
                    raise ValueError(f'Argument {i} out of range [{mn}, {mx}]')
            return func(*args)
        return wrap
    return checker

@rangecheck(0, 255)
def color(r, g, b):
    return f'RGB({r}, {g}, {b})'

print(color(10, 200, 50))    # Works
print(color(0, 128, 34))     # Works

try:
    print(color(0, 256, 0))  # 256 > 255 → ValueError
except ValueError as e:
    print('Error:', e)

print()

# Manual equivalent (shows the 3 steps)
deco = rangecheck(0, 255)    # Step 1: call factory
wrapped = deco(color)        # Step 2: apply decorator
# @rangecheck(0,255) does both automatically

---
## 9. 🏗️ `__new__` vs `__init__`

### Theory

**Object creation is a TWO-step process:**

1. **`__new__(cls, ...)`** — *allocates* and *returns* the new object. A static method; receives the class `cls`. This is where memory is allocated.
2. **`__init__(self, ...)`** — *initializes* the already-created object. Receives the instance. Returns `None`.

**Normal usage**: you never override `__new__` unless implementing Singleton, Flyweight, or subclassing immutables (`int`, `str`, `tuple` — because `__init__` can't change the value of an already-created immutable).

⚠️ **If `__new__` returns something that is NOT an instance of `cls`, then `__init__` is NOT called.**

In [ ]:
class AClass:
    def __new__(cls, *args, **kwargs):
        print('__new__ called — allocating')
        ob = object.__new__(cls)   # Actually allocate
        return ob

    def __init__(self, *args, **kwargs):
        print(f'__init__ called — args={args}, kwargs={kwargs}')

print('--- AClass() ---')
o1 = AClass()

print('--- AClass(1, 2) ---')
o2 = AClass(1, 2)

print('--- AClass(name="John") ---')
o3 = AClass(name='John')

print()

# Singleton pattern using __new__
class Singleton:
    _instance = None
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance  # Always returns the SAME object

s1 = Singleton()
s2 = Singleton()
print(f's1 is s2 = {s1 is s2}')  # True!

---
## 10. 🔐 Name Mangling — `__private`

### Theory

**Python has NO truly private attributes. It uses name mangling as a convention.**

- **`_name`** (single underscore) — convention meaning "internal/protected". No mangling. Accessible normally.
- **`__name`** (double underscore prefix, at most one trailing underscore) — Python **rewrites** this to `_ClassName__name` at compile time. Purpose: avoid accidental collisions in subclasses.

⚠️ **`__name` is NOT truly private!** From outside the class, `obj.__name` raises `AttributeError` (Python looks for `__name` literally), but `obj._ClassName__name` works fine.

⚠️ **Dunder names like `__init__`, `__str__` are NOT mangled** (they have two underscores on BOTH sides).

In [ ]:
class Person:
    def __init__(self, name):
        self.__check(name)    # stored as _Person__check
        self.__name = name    # stored as _Person__name

    def getName(self): return self.__name
    def setName(self, name):
        self.__check(name)
        self.__name = name

    def __check(self, s):
        if not s.isalpha():
            raise ValueError(f'{s!r} contains non-alpha chars')

john = Person('Bob')
print(john.getName())      # Bob
john.setName('John')
print(john.getName())      # John

# Direct access → AttributeError
try:
    print(john.__name)
except AttributeError as e:
    print('AttributeError:', e)

# Mangled name is accessible!
print('Via mangled name:', john._Person__name)  # John

print()
# All actual attribute names
print('vars(john):', vars(john))

---
## 11. ✨ Magic Methods: `__eq__`, `__hash__`, `__bool__`, `__call__`, `__radd__`

### Theory

**`__eq__` and `__hash__` — the inseparable pair:**
**If you define `__eq__`, Python automatically sets `__hash__ = None`** (making instances unhashable). This enforces the invariant: *equal objects must have equal hashes*. If you define custom equality, you MUST also define custom hashing.

⚠️ **Objects are usable as dict keys / set members ONLY if they are hashable.**

**`__bool__`** — called by `bool(obj)` and `if obj:`. If not defined, Python tries `__len__` (truthy if non-zero), then defaults to `True`.

**`__call__`** — makes instances callable: `obj(args)` → `obj.__call__(args)`.

**Reflected operators (`__radd__`, `__rmul__`, etc.):**
- `a + b` first tries `a.__add__(b)`. If that returns `NotImplemented` → tries `b.__radd__(a)`.
- Allows `5 + my_obj` to work even though `int` doesn't know your class.

In [ ]:
# __eq__ + __hash__ — needed for dict keys / set membership
class Point:
    def __init__(self, x, y): self.x, self.y = x, y
    def __eq__(self, other): return (self.x, self.y) == (other.x, other.y)
    def __hash__(self): return hash((self.x, self.y))   # tuple hash is stable
    def __repr__(self): return f'Point({self.x},{self.y})'

p1 = Point(1, 2)
p2 = Point(1, 2)   # Same value, different object
p3 = Point(3, 4)

print(f'p1 == p2: {p1 == p2}')    # True  (equal values)
print(f'p1 is p2: {p1 is p2}')    # False (different objects)

# Usable as dict key!
d = {Point(1,2): 'origin', Point(3,4): 'other'}
print(d[p1])   # 'origin' — p1 == Point(1,2) so lookup works!

# Set deduplication
s = {p1, p2, p3}
print(f'Set {{p1, p2, p3}} size: {len(s)}')  # 2  (p1 and p2 are equal → deduplicated)

In [ ]:
# __bool__ — custom truthiness
class Person:
    def __init__(self, name, age): self.name, self.age = name, age
    def __repr__(self): return f'Person({self.name},{self.age})'
    def __bool__(self): return self.age >= 18   # Adults are truthy

people = [Person('Alice',13), Person('Bob',19), Person('Charlie',10), Person('Dave',25)]
adults = [p for p in people if p]   # Uses __bool__ implicitly!
print('Adults:', adults)

print()

# __call__ — callable instances
class AgeFilter:
    def __init__(self, low, high): self.low, self.high = low, high
    def __call__(self, p): return self.low <= p.age <= self.high

is_teen = AgeFilter(13, 17)
is_adult = AgeFilter(18, 120)
print('Teens:', [p for p in people if is_teen(p)])
print('Adults:', [p for p in people if is_adult(p)])

print()

# __radd__ — reflected addition
class Mod7:
    def __init__(self, n): self.n = n % 7
    def __add__(self, v):  return Mod7(self.n + v)
    def __radd__(self, v): return Mod7(self.n + v)  # Called when int+Mod7!
    def __repr__(self): return f'Mod7({self.n})'

m = Mod7(4)
print(m + 5)   # Mod7(2)  — m.__add__(5)
print(4 + m)   # Mod7(1)  — int.__add__(m) fails → m.__radd__(4) called!

---
## 12. 📐 `@functools.total_ordering`

### Theory

**To make a class fully orderable you need all 6 comparison operators.** `@functools.total_ordering` generates the missing ones from just `__eq__` + one of `__lt__`/`__le__`/`__gt__`/`__ge__`.

⚠️ **Without `total_ordering`, if you only define `__lt__`, calling `<=`, `>`, `>=` raises `TypeError`.** With it, all 6 operators are derived automatically.

In [ ]:
import functools

@functools.total_ordering
class ByLength:
    """Sorts strings first by length, then alphabetically."""
    def __init__(self, s): self.s = s
    def __repr__(self): return self.s

    def __eq__(self, other):
        return self.s.lower() == other.s.lower()

    def __lt__(self, other):
        if len(self.s) != len(other.s):
            return len(self.s) < len(other.s)
        return self.s.lower() < other.s.lower()
    # total_ordering derives __le__, __gt__, __ge__, __ne__ automatically!

words = [ByLength('banana'), ByLength('fig'), ByLength('apple'), ByLength('kiwi'), ByLength('plum')]
print('Unsorted:', words)
words.sort()
print('Sorted:  ', words)

a, b = ByLength('hi'), ByLength('bye')
print(f'hi < bye: {a<b}, hi <= bye: {a<=b}, hi >= bye: {a>=b}, hi > bye: {a>b}')

---
## 13. 🔀 `@singledispatch` — Dispatch Rules

### Theory

**`@functools.singledispatch`** creates a generic function that dispatches based on the **type of the first argument**, following the MRO.

**Critical exam rules:**

⚠️ **`bool` is a subclass of `int`** — `func(True)` dispatches to the `int` handler!

⚠️ **`Dog(Animal)` subclass** — if no `Dog` handler but there IS an `Animal` handler, `func(Dog(...))` dispatches to `Animal` handler.

⚠️ **Union types** (`list | tuple`) — register one handler for multiple types.

⚠️ **No matching handler** → falls back to the base `@singledispatch` function (the "default").

In [ ]:
import functools

class Animal:
    def __init__(self, name): self.name = name
    def __repr__(self): return f'Animal({self.name})'

class Dog(Animal):
    def __repr__(self): return f'Dog({self.name})'

@functools.singledispatch
def describe(arg):
    raise NotImplementedError(f'No handler for {type(arg).__name__}')

@describe.register(int)
def _(v): print(f'int handler: {v} * 2 = {v * 2}')

@describe.register(str)
def _(v): print(f'str handler: {v.upper()!r}')

@describe.register(list | tuple)  # Union!
def _(v): print(f'list/tuple handler: length={len(v)}')

@describe.register(Animal)        # Handles Animal AND subclasses
def _(v): print(f'Animal handler: {v}')

describe(5)              # int handler
describe('hello')        # str handler
describe([1,2,3])        # list|tuple handler
describe((4,5,6))        # list|tuple handler

print()
print('KEY EXAM TRAPS:')
describe(True)           # → int handler! (bool IS-A int)
describe(Dog('Rex'))     # → Animal handler! (Dog IS-A Animal)

try:
    describe(3.14)       # float → no handler → NotImplementedError
except NotImplementedError as e:
    print(f'NotImplementedError: {e}')

---
## 14. 🎯 Exception Mechanics — `else`, Control Flow, `exception.args`

### Theory

**`try/except/else/finally`:**
- `else` — runs **only if NO exception was raised** in the `try` block
- `finally` — runs **always**, even with `return` or `raise` in `try`

**Exceptions as control flow:**
Your lecture shows `raise Exception(a, b, c)` to **escape all nested loops at once** and pass back values via `ex.args`.

⚠️ **`ex.args`** is a tuple of the arguments passed to the constructor: `Exception(1,2,3).args == (1,2,3)`.

**`for/while...else`:**
The `else` block runs **only if the loop completed WITHOUT a `break`** — perfect for "not found" patterns.

In [ ]:
# try/except/else/finally
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print('Division by zero!')
        return None
    else:
        print(f'Success: {a}/{b} = {result}')  # Only if no exception
        return result
    finally:
        print('Finally — always runs')         # Always!

print('--- 10/2 ---'); safe_divide(10, 2)
print()
print('--- 5/0 ---'); safe_divide(5, 0)

In [ ]:
# EXCEPTION AS CONTROL FLOW — escape nested loops
lst = [9, 1, 5, 7, 11, 10, 4]
target = 20

try:
    for a in range(len(lst)-2):
        for b in range(a+1, len(lst)-1):
            for c in range(b+1, len(lst)):
                if lst[a] + lst[b] + lst[c] == target:
                    raise Exception(a, b, c)   # Escape all 3 loops!
except Exception as ex:
    indices = ex.args       # (a, b, c) as a TUPLE
else:
    indices = (-1, -1, -1)  # Only if loop never raised

if indices != (-1,-1,-1):
    a, b, c = indices
    print(f'Found indices {indices}: {lst[a]}+{lst[b]}+{lst[c]}={target}')
else:
    print('Not found')

print()

# for...else — else runs only if no break
def find_prime(numbers):
    for n in numbers:
        for i in range(2, int(n**0.5)+1):
            if n % i == 0:
                break          # Not prime
        else:
            return n           # Loop completed without break → n is prime!
    return None

print('First prime in [4,6,8,13,15]:', find_prime([4,6,8,13,15]))  # 13
print('First prime in [4,6,8,9,15]: ', find_prime([4,6,8,9,15]))   # None

---
## 15. 🤔 `and`/`or` Return Values, Not Booleans

### Theory

**Python's `and` and `or` use short-circuit evaluation and return one of their OPERANDS — not necessarily `True`/`False`.**

- **`a or b`** — returns `a` if `a` is truthy, otherwise returns `b`
- **`a and b`** — returns `a` if `a` is falsy, otherwise returns `b`

**Falsy values**: `None`, `0`, `0.0`, `''`, `[]`, `{}`, `set()`, `False`

**`not` always returns a proper `bool`.**

⚠️ **`[] or 'default'` returns `'default'` (the string, not `True`).**
⚠️ **`[1] and 'hello'` returns `'hello'` (not `True`).**

In [ ]:
# or — returns first TRUTHY value, or the LAST value
print('--- or ---')
print([] or 'default')     # 'default'  ([] is falsy)
print([1] or 'default')    # [1]        ([1] is truthy)
print(0 or False or None)  # None       (all falsy → last)
print(0 or 5 or 10)        # 5          (first truthy)

# and — returns first FALSY value, or the LAST value
print('--- and ---')
print([1] and 'hello')     # 'hello'    ([1] truthy → continue → last)
print([] and 'hello')      # []         ([] falsy → stop → return [])
print(1 and 2 and 3)       # 3          (all truthy → last)
print(1 and 0 and 3)       # 0          (0 is first falsy)

# not — always returns bool
print('--- not ---')
print(not [])              # True
print(not [1, 2])          # False
print(type(not []))        # <class 'bool'>

print()

# Practical idiom: default values
name = '' or 'Anonymous'
print('Name:', name)           # Anonymous

data = [1, 2, 3]
first = data and data[0]       # Safe: only access [0] if data is truthy
print('First element:', first) # 1

---
## 16. ⛓️ Chained Comparisons

### Theory

**Python supports chained comparisons:** `a < b < c` is equivalent to `a < b and b < c`, but `b` is evaluated **only once**.

⚠️ **`a != b != c` does NOT mean all three are different!** It means `a != b AND b != c`. So `1 != 2 != 1` is `True` even though `1` appears on both sides.

In [ ]:
x = 5
print(1 < x < 10)    # True
print(1 < x < 5)     # False
print(1 < x <= 5)    # True
print()

# != trap
print(1 != 2 != 1)   # True!  (1!=2 is True AND 2!=1 is True) — but 1 appears twice!
print(1 != 2 != 3)   # True
print(1 != 1 != 2)   # False  (1!=1 is False)
print()

# Side effect: b evaluated only once
def middle():
    print('  middle() called')
    return 5

print('In chain 1 < middle() < 10:')
result = 1 < middle() < 10   # middle() called ONCE
print(f'Result: {result}')

---
## 17. 📋 List Multiplication — Shallow References!

### Theory

**`[x] * n` creates a list of `n` references to the same object `x`.**

For **immutables** (int, str, tuple): this is safe — you can't mutate `0` anyway.

For **mutables** (list, dict): all `n` slots point to the **SAME inner object**. Mutating any one mutates all!

⚠️ **Classic 2D grid trap**: `grid = [[0]*3] * 3` — all 3 rows are literally the SAME list object.

In [ ]:
# Safe — immutables
nums = [0] * 5
nums[2] = 99
print('nums:', nums)   # [0, 0, 99, 0, 0]  — fine
print()

# TRAP — mutable inner list
bad_grid = [[0]*3] * 3
print('bad_grid before:', bad_grid)
bad_grid[0][1] = 99    # Modify "row 0"
print('bad_grid after: ', bad_grid)  # ALL 3 rows changed!
print('Same inner list:', bad_grid[0] is bad_grid[1])  # True!
print()

# FIX — list comprehension creates independent lists
good_grid = [[0]*3 for _ in range(3)]
print('good_grid before:', good_grid)
good_grid[0][1] = 99
print('good_grid after: ', good_grid)  # Only row 0 changed!
print('Same inner list:', good_grid[0] is good_grid[1])  # False!

---
## 18. 📦 Extended Unpacking — Starred Assignment

### Theory

**The starred `*var` collects remaining elements into a list.** Only ONE starred variable allowed per unpacking expression.

⚠️ **The starred variable ALWAYS captures a list** — even if it gets zero or one element.

In [ ]:
a, *b = [1, 2, 3, 4]
print(f'a={a}, b={b}')         # a=1, b=[2, 3, 4]

*a, b = [1, 2, 3, 4]
print(f'a={a}, b={b}')         # a=[1, 2, 3], b=4

a, *b, c = [1, 2, 3, 4, 5]
print(f'a={a}, b={b}, c={c}')  # a=1, b=[2, 3, 4], c=5

# Starred gets [] if nothing remains
a, *b, c = [1, 2]
print(f'a={a}, b={b}, c={c}')  # a=1, b=[], c=2  — b is empty list!

print()

# Swap in one line — RHS fully evaluated before assignment
x, y = 10, 20
x, y = y, x
print(f'After swap: x={x}, y={y}')  # x=20, y=10

print()

# Unpacking in for loops
pairs = [(1,'a'), (2,'b'), (3,'c')]
for num, letter in pairs:
    print(f'{num}→{letter}', end='  ')

---
## 19. 🔢 Boolean Arithmetic — `True=1`, `False=0`

### Theory

**`bool` is a subclass of `int`.** `True` IS `1` and `False` IS `0` for arithmetic purposes.

- `True + True` = `2`
- `True * 5` = `5`
- `sum([True, False, True, True])` = `3` — elegant way to count `True` values

⚠️ **In sets and dicts: `True` and `1` are treated as equal keys. So `{True, 1}` has only ONE element!**

In [ ]:
print('True + True  =', True + True)    # 2
print('True * 5     =', True * 5)       # 5
print('False * 99   =', False * 99)     # 0
print()

# Count Trues
bools = [True, False, True, True, False, True]
print('Count of True:', sum(bools))     # 4
print()

print('issubclass(bool, int):', issubclass(bool, int))  # True
print('isinstance(True, int):', isinstance(True, int))  # True
print()

# True==1 but True is not 1
print('True == 1:  ', True == 1)    # True
print('True is 1:  ', True is 1)    # False!
print()

# Sets/dicts treat True==1 and False==0 as SAME KEY
s = {True, 1, False, 0}
print('Set {True, 1, False, 0}:', s)        # {False, True}  — 2 elements!

d = {True: 'yes', 1: 'one'}
print('Dict {True:"yes", 1:"one"}:', d)     # {True: 'one'}  — key overwritten!

---
## 20. 🌊 Pull-pipe vs Push-pipe Generator Pipelines

### Theory

**Two complementary pipeline architectures in your course:**

**Pull-pipe (lazy generator chain):**
- Each stage is a **generator function** (uses `yield`)
- Data flows by pulling: consumer calls `next()` which pulls from upstream
- Composed by passing upstream generator as argument
- Notation: `toList(mapp(filt(limi(src))))` — reads inside-out (source innermost)

**Push-pipe (coroutine chain):**
- Each stage is a **primed coroutine** (receives via `send()`)
- Data flows by pushing: source calls `target.send(item)` to push downstream
- Composed by passing downstream coroutine as argument
- Notation: `limi(filt(mapp(sink)))` — reads outside-in (source is separate)

In [ ]:
# PULL PIPE — lazy generator chain
def limi_pull(n, source):           # Limit to n items
    for _ in range(n):
        yield next(source)

def filt_pull(pred, source):        # Filter
    for item in source:
        if pred(item):
            yield item

def mapp_pull(func, source):        # Map
    for item in source:
        yield func(item)

src = iter(range(1, 20))
pred = lambda n: n % 2 == 0        # Even numbers
func = lambda x: x * 10

# Pipeline composition — reads inside-out
pipeline = mapp_pull(func, filt_pull(pred, limi_pull(5, src)))
print('Pull-pipe result:', list(pipeline))  # 10 items limited, evens filtered, *10

# Data flows lazily: list() triggers execution by pulling from mapp,
# which pulls from filt, which pulls from limi, which pulls from src

---
## 21. 🎁 Properties — Getters, Setters, Validation

### Theory

**Properties let you define methods that are accessed like attributes.** Add validation, computation, or side effects without changing the public interface.

⚠️ **The getter, setter, and deleter must all have the SAME name as the property.**

⚠️ **`self.res = r` in `__init__` CALLS THE SETTER.** So validation runs even at construction time. Internally, properties store the actual value in `self._res` (single underscore) to avoid infinite recursion.

In [ ]:
class Resistor:
    def __init__(self, r=0):
        self.res = r   # ← This CALLS the @res.setter!

    @property
    def res(self):
        """Resistance value."""
        return self._res

    @res.setter
    def res(self, r):
        if r < 0: raise ValueError('Negative resistance')
        self._res = float(r)   # Stored as float

    def __add__(self, other):    # Series combination
        return Resistor(self.res + other.res)

    def __mul__(self, other):    # Parallel combination
        return Resistor(self.res * other.res / (self.res + other.res))

    def __repr__(self):
        return f'Resistor({self.res:.2f}Ω)'

r1, r2 = Resistor(3), Resistor(6)
print(f'Series:   {r1} + {r2} = {r1 + r2}')   # 9Ω
print(f'Parallel: {r1} * {r2} = {r1 * r2}')   # 2Ω
print()

# Validation at construction time
try:
    bad = Resistor(-5)   # setter called in __init__ → ValueError!
except ValueError as e:
    print('Caught at construction:', e)

# Validation when setting
try:
    r1.res = -3
except ValueError as e:
    print('Caught on assignment:', e)

---
## 22. 🧬 MRO — Multiple Inheritance Resolution Order

### Theory

**Python uses the C3 linearization algorithm to determine the MRO** — the order to search base classes when looking up a method.

**`super()`** calls the NEXT class in the MRO, not necessarily the direct parent.

⚠️ **Diamond inheritance** — without `super()`, a base class method could be called multiple times. With cooperative `super()` calls, it's called exactly once.

In [ ]:
class A:
    def method(self): print('A.method')

class B(A):
    def method(self):
        print('B.method')
        super().method()   # Next in MRO after B

class C(A):
    def method(self):
        print('C.method')
        super().method()

class D(B, C):
    def method(self):
        print('D.method')
        super().method()

print('MRO of D:', [c.__name__ for c in D.__mro__])
# D → B → C → A → object
print()
print('Calling D().method():')
D().method()
# D → B → C → A  — A.method called ONCE, not twice (that's the point of super()!)

---
## 23. 🎯 Quick-Fire Exam Traps — "What Does This Print?"

In [ ]:
# TRAP 1: Mutable default — list grows across calls
def add(item, lst=[]):
    lst.append(item)
    return lst

r1 = add(1)
r2 = add(2)
print('Trap 1:', r1, r2)   # [1, 2]  [1, 2]  — SAME list object!

In [ ]:
# TRAP 2: Late binding in lambdas
funcs = [lambda: i for i in range(3)]
print('Trap 2:', [f() for f in funcs])   # [2, 2, 2]  NOT [0, 1, 2]!

In [ ]:
# TRAP 3: UnboundLocalError
n = 10
def f():
    try:
        print(n)  # UnboundLocalError — n is local because of n=5 below!
    except UnboundLocalError as e:
        print('Trap 3:', type(e).__name__)
    n = 5
f()

In [ ]:
# TRAP 4: Shallow copy of nested list
a = [[1, 2], [3, 4]]
b = a[:]            # shallow copy
b[0][0] = 99
print('Trap 4:', a)  # [[99, 2], [3, 4]]  — a is affected!

In [ ]:
# TRAP 5: Decorator execution time
def deco(f):
    print(f'Decorating {f.__name__}')  # Runs NOW at decoration
    def wrapper(): return f()
    return wrapper

print('--- Before @deco ---')
@deco
def my_func(): return 42
print('--- After @deco, before call ---')
print('Result:', my_func())

In [ ]:
# TRAP 6: bool dispatches to int handler in singledispatch
import functools

@functools.singledispatch
def handle(v): return f'default'

@handle.register(int)
def _(v): return f'int: {v}'

print('Trap 6:', handle(True))   # 'int: True'  — bool IS-A int!

In [ ]:
# TRAP 7: True and 1 are the same dict key
d = {True: 'yes', 1: 'one', False: 'no', 0: 'zero'}
print('Trap 7:', d)   # {True: 'one', False: 'zero'}  — only 2 keys!

In [ ]:
# TRAP 8: Generator exhaustion
def gen():
    yield 1; yield 2; yield 3

g = gen()
print('Trap 8a:', list(g))   # [1, 2, 3]
print('Trap 8b:', list(g))   # []  — exhausted!
print('Trap 8c:', list(gen()))  # [1, 2, 3]  — fresh generator from function call

In [ ]:
# TRAP 9: and/or return values not booleans
print('Trap 9a:', 0 or 'hello')    # 'hello'
print('Trap 9b:', [] and 'hello')  # []
print('Trap 9c:', 5 and 'hello')   # 'hello'
print('Trap 9d:', type(not []))    # <class 'bool'>  — not always returns bool

In [ ]:
# TRAP 10: List multiplication
grid = [[0]*3] * 3
grid[0][0] = 9
print('Trap 10:', grid)   # [[9,0,0],[9,0,0],[9,0,0]] — all rows same object!

In [ ]:
# TRAP 11: Exception args as tuple
try:
    raise Exception(1, 2, 3)
except Exception as e:
    print('Trap 11: e.args =', e.args)        # (1, 2, 3)  — tuple!
    print('         type   =', type(e.args))   # <class 'tuple'>

In [ ]:
# TRAP 12: != chaining
print('Trap 12a:', 1 != 2 != 1)   # True!  but 1 appears twice — NOT all different!
print('Trap 12b:', 1 != 1 != 2)   # False  (1!=1 is False)

# TRAP 13: for...else
for x in [2, 4, 6]:
    if x % 2 != 0:
        break
else:
    print('Trap 13: else ran — no break occurred')  # This runs!

In [ ]:
# TRAP 14: __new__ returns non-instance → __init__ not called
class Weird:
    def __new__(cls):
        print('__new__ called')
        return 42   # NOT an instance of Weird!
    def __init__(self):
        print('__init__ called')  # This will NOT run!

w = Weird()
print('Trap 14: w =', w, type(w))  # 42 <class 'int'>  — __init__ never called!

In [ ]:
# TRAP 15: Name mangling with subclasses
class Base:
    def __init__(self):
        self.__x = 'base_x'   # stored as _Base__x

class Child(Base):
    def __init__(self):
        super().__init__()
        self.__x = 'child_x'  # stored as _Child__x  — DIFFERENT attribute!

c = Child()
print('Trap 15: vars =', vars(c))
# {'_Base__x': 'base_x', '_Child__x': 'child_x'}  — two separate attributes!

---

## 📋 Master Reference Card

| Topic | What surprises people | The rule |
|---|---|---|
| **`is` vs `==`** | `a is b` checks identity, not value | Use `==` for values; `is` only for `None`/singletons |
| **Integer cache** | `-5..256` reuse same object | `257 is 257` may be False in REPL |
| **Mutable default** | `def f(lst=[])` — list shared across calls | Use `lst=None`, create inside function |
| **Default eval time** | `def f(b=i)` — captures `i` at definition | Default frozen at definition, not at call |
| **Late binding** | `lambda: i` captures variable, not value | Use `lambda i=i: i` to fix |
| **UnboundLocalError** | Any assignment in function makes var local for WHOLE function | Add `global`/`nonlocal` if you need to rebind outer |
| **`__new__` vs `__init__`** | `__new__` allocates, `__init__` initializes | `__init__` skipped if `__new__` returns non-instance |
| **Name mangling** | `__x` → `_ClassName__x` | Not truly private — accessible via mangled name |
| **`__eq__` kills `__hash__`** | Defining `__eq__` sets `__hash__=None` | Always define both together |
| **`__radd__`** | `4 + my_obj` → `my_obj.__radd__(4)` | Define for right-side operations |
| **singledispatch** | `bool` → `int` handler; subclass → parent handler | Dispatch follows MRO |
| **Decorator order** | `@A @B def f` → `f = A(B(f))` | Applied bottom-up, outer executes first at call time |
| **Decoration time** | Outer decorator body runs at import, not call | `print()` in decorator runs when module loads |
| **Generator `send()`** | Must prime before `send(non-None)` | Use priming decorator or `next()` first |
| **Generator exhaustion** | Iterating a generator twice — second time is empty | `list(gen())` not `list(g)` twice |
| **`or`/`and`** | Return VALUES, not booleans | `[] or 'x'` = `'x'`, `[1] and 'x'` = `'x'` |
| **`for...else`** | `else` runs only if NO `break` | Perfect for "not found" patterns |
| **`try...else`** | `else` runs only if NO exception | Different from `finally` (which always runs) |
| **exception.args** | `raise Exception(a,b,c)` → `e.args == (a,b,c)` tuple | Control flow across nested loops |
| **List multiplication** | `[[]] * n` → n refs to SAME list | Use `[[] for _ in range(n)]` |
| **Shallow copy** | `list(a)` — new outer, same inner objects | `deepcopy` for full independence |
| **`bool` is `int`** | `True==1`, `isinstance(True,int)` is True | `{True,1}` has 1 element; `sum([True,False,True])==2` |
| **`!=` chaining** | `1 != 2 != 1` is True (≠ all different) | Means `1!=2 AND 2!=1`, not "all distinct" |
| **Name mangling + subclass** | `__x` in Child → `_Child__x`, not `_Base__x` | Each class gets its own mangled name |